# Attempt 002: GPU and model access
A short environment check before fine-tuning.

In [ ]:
import json, platform, importlib.metadata, time
from pathlib import Path
import torch
result = {
    'python': platform.python_version(), 'torch': torch.__version__,
    'cuda': torch.version.cuda, 'cuda_available': torch.cuda.is_available(),
    'gpu_count': torch.cuda.device_count(),
    'packages': {}, 'gpus': [], 'model_configs': [],
}
for name in ['transformers', 'peft', 'accelerate', 'bitsandbytes', 'datasets', 'flash-attn']:
    try: result['packages'][name] = importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError: result['packages'][name] = None
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    result['gpus'].append({'index': i, 'name': p.name, 'memory_bytes': p.total_memory,
                           'capability': list(torch.cuda.get_device_capability(i))})
# Search only the deliberately attached Kaggle input directory; no local/FUSE traversal.
for parent in [Path('/kaggle/input/gemma-2'), Path('/kaggle/input/models/google/gemma-2')]:
    if parent.exists():
        for p in parent.glob('transformers/*/*/config.json'):
            c = json.loads(p.read_text())
            result['model_configs'].append({'path': str(p.parent), 'model_type': c.get('model_type'),
                'hidden_size': c.get('hidden_size'), 'num_hidden_layers': c.get('num_hidden_layers')})
if torch.cuda.is_available():
    a = torch.randn(2048, 2048, device='cuda', dtype=torch.float16, requires_grad=True)
    torch.cuda.synchronize(); start = time.monotonic()
    loss = (a @ a.T).float().square().mean(); loss.backward(); torch.cuda.synchronize()
    result['gpu_backward_test'] = {'finite_loss': bool(torch.isfinite(loss)), 'seconds': time.monotonic()-start}
Path('/kaggle/working/gpu_probe.json').write_text(json.dumps(result, indent=2))
print(json.dumps(result, indent=2))
assert result['cuda_available'], 'The requested GPU was not allocated'
assert result['gpu_backward_test']['finite_loss'], 'GPU arithmetic check failed'
